In [1]:
import torch
print(torch.__version__)

2.11.0


Use autograd to find the gradient vector of f(x, y) = sin(x2 y) at the
point (x, y) = (1.2, 3.4).

In [2]:

x = torch.tensor(1.2, requires_grad=True)
y = torch.tensor(3.4, requires_grad=True)
f = torch.sin(x**2 * y)
f.backward()
grad = torch.tensor([x.grad, y.grad])  # or use x.grad.item(), y.grad.item()
print(grad)

tensor([1.4899, 0.2629])


14. Create a custom Dense module that replicates the functionality of
a nn.Linear module followed by a nn.ReLU module. Try
implementing it first using the nn.Linear and nn.ReLU
modules, and then reimplement it using nn.Parameter and the
relu() function.

In [3]:
import torch
import torch.nn as nn
class Dense(nn.Module):
    """Linear layer followed by ReLU (same idea as nn.Linear + nn.ReLU)."""
    def __init__(self, in_features: int, out_features: int, bias: bool = True):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features, bias=bias)
        self.relu = nn.ReLU()
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.relu(self.linear(x))

In [ ]:
import math
import torch.nn as nn
import torch.nn.functional as F
class Dense(nn.Module):
    """y = ReLU(x @ W^T + b), matching nn.Linear(in, out) + ReLU."""
    def __init__(self, in_features: int, out_features: int):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.weight = nn.Parameter(torch.empty(out_features, in_features))
        self.bias = nn.Parameter(torch.empty(out_features))
        
        self.reset_parameters()

    def reset_parameters(self) -> None:
        nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))
        fan_in, _ = nn.init._calculate_fan_in_and_fan_out(self.weight)
        bound = 1 / math.sqrt(fan_in) if fan_in > 0 else 0
        nn.init.uniform_(self.bias, -bound, bound)
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return F.relu(F.linear(x, self.weight, self.bias))

15. Build and train a classification MLP on the CoverType dataset:

a. Load the dataset using
sklearn.datasets.fetch_covtype() and create
a custom PyTorch Dataset for this data.

In [9]:
import torch
from torch.utils.data import Dataset
from sklearn.datasets import fetch_covtype

class CovtypeDataset(Dataset):
    """PyTorch Dataset for the UCI Covertype data from sklearn."""
    def __init__(self, X=None, y=None, *, download: bool = True):
        if X is None or y is None:
            bunch = fetch_covtype(download=download)
            X = bunch.data
            y = bunch.target - 1  # sklearn uses 1..7; CrossEntropyLoss wants 0..6
        self.X = torch.as_tensor(X, dtype=torch.float32)
        self.y = torch.as_tensor(y, dtype=torch.long)
    def __len__(self):
        return self.X.shape[0]
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [10]:
import numpy as np
import torch
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from sklearn.datasets import fetch_covtype
from sklearn.preprocessing import StandardScaler

bunch = fetch_covtype()
X = bunch.data.astype("float32")
y = bunch.target.astype("int64") - 1  # 0..6 for CrossEntropyLoss
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.15 / 0.85, random_state=42, stratify=y_trainval
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

train_ds = CovtypeDataset(X_train, y_train)
val_ds = CovtypeDataset(X_val, y_val)
test_ds = CovtypeDataset(X_test, y_test)
X_trainval_scaled = np.vstack([X_train, X_val])
y_trainval = np.concatenate([y_train, y_val])
trainval_ds = CovtypeDataset(X_trainval_scaled, y_trainval)

batch_size = 256
pin_memory = torch.cuda.is_available()

train_loader = DataLoader(
    train_ds, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=pin_memory
)
val_loader = DataLoader(
    val_ds, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=pin_memory
)
test_loader = DataLoader(
    test_ds, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=pin_memory
)
trainval_loader = DataLoader(
    trainval_ds, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=pin_memory
)

c. Build a custom MLP module to tackle this classification
task. You can optionally use the custom Dense module
from the previous exercise.

In [11]:
import torch
import torch.nn as nn


class Dense(nn.Module):
    """Linear layer followed by ReLU (same idea as nn.Linear + nn.ReLU)."""

    def __init__(self, in_features: int, out_features: int, bias: bool = True):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features, bias=bias)
        self.relu = nn.ReLU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.relu(self.linear(x))


class MLP(nn.Module):
    """MLP for Covertype: 54 features -> hidden Dense blocks -> 7 logits."""

    def __init__(
        self,
        in_features: int = 54,
        num_classes: int = 7,
        hidden_sizes: tuple[int, ...] = (128, 64),
        bias: bool = True,
        dropout: float = 0.0,
    ):
        super().__init__()
        layers: list[nn.Module] = []
        prev = in_features
        for h in hidden_sizes:
            layers.append(Dense(prev, h, bias=bias))
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        self.hidden = nn.Sequential(*layers)
        self.head = nn.Linear(prev, num_classes, bias=bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.hidden(x)
        return self.head(x)

d. Train this model on the GPU, and try to reach 93%
accuracy on the test set. For this, you will likely have to
perform hyperparameter search to find the right number of
layers and neurons per layer, a good learning rate and batch size, and so on. 

You can optionally use Optuna for this.

In [12]:
# Requires: pip install optuna
import numpy as np
import torch.nn as nn
import optuna
from optuna.pruners import MedianPruner
from sklearn.utils.class_weight import compute_class_weight

# Run the Covertype cell that defines train_loader, val_loader, test_loader, trainval_loader.
for _name in ("train_loader", "val_loader", "test_loader", "trainval_loader"):
    assert _name in globals(), f"Run the data loader cell first (missing {_name})."

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pin = device.type == "cuda"
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)


def train_one_epoch(model, loader, optimizer, criterion) -> None:
    model.train()
    for xb, yb in loader:
        xb = xb.to(device, non_blocking=pin)
        yb = yb.to(device, non_blocking=pin)
        optimizer.zero_grad(set_to_none=True)
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()


@torch.no_grad()
def validation_accuracy(model, loader) -> float:
    model.eval()
    correct = 0
    total = 0
    for xb, yb in loader:
        xb = xb.to(device, non_blocking=pin)
        yb = yb.to(device, non_blocking=pin)
        logits = model(xb)
        correct += (logits.argmax(dim=1) == yb).sum().item()
        total += yb.numel()
    return correct / total


y_train_np = np.asarray(y_train)
class_weights = compute_class_weight("balanced", classes=np.arange(7), y=y_train_np)
cw_tensor = torch.tensor(class_weights, dtype=torch.float32, device=device)


def objective(trial: optuna.Trial) -> float:
    n_layers = trial.suggest_int("n_layers", 2, 5)
    hidden_sizes = tuple(
        trial.suggest_int(f"n_units_{i}", 96, 640, log=True) for i in range(n_layers)
    )
    lr = trial.suggest_float("lr", 3e-5, 3e-3, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 5e-2, log=True)
    dropout = trial.suggest_float("dropout", 0.0, 0.45)

    model = MLP(
        in_features=54,
        num_classes=7,
        hidden_sizes=hidden_sizes,
        dropout=dropout,
    ).to(device)
    
    criterion = nn.CrossEntropyLoss(weight=cw_tensor)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.5, patience=2, min_lr=1e-6
    )

    max_epochs = 60
    best_val = 0.0
    patience_es = 12
    stalled = 0

    for epoch in range(max_epochs):
        train_one_epoch(model, train_loader, optimizer, criterion)
        val_acc = validation_accuracy(model, val_loader)
        scheduler.step(val_acc)
        trial.report(val_acc, epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()

        if val_acc > best_val + 1e-6:
            best_val = val_acc
            stalled = 0
        else:
            stalled += 1
        if stalled >= patience_es:
            break

    return best_val


study = optuna.create_study(
    direction="maximize",
    pruner=MedianPruner(n_startup_trials=5, n_warmup_steps=8),
)
study.optimize(objective, n_trials=40, show_progress_bar=True)

print("Best trial:")
print(study.best_trial.value, study.best_trial.params)

# --- Final model: train on train+val via trainval_loader, evaluate with test_loader ---
best = study.best_trial.params
n_layers = best["n_layers"]
hidden_sizes = tuple(best[f"n_units_{i}"] for i in range(n_layers))
lr = best["lr"]
weight_decay = best["weight_decay"]
dropout = best["dropout"]

y_trainval = np.concatenate([y_train, y_val])
cw_final = torch.tensor(
    compute_class_weight("balanced", classes=np.arange(7), y=y_trainval),
    dtype=torch.float32,
    device=device,
)

final_model = MLP(in_features=54, num_classes=7, hidden_sizes=hidden_sizes, dropout=dropout).to(device)
criterion_f = nn.CrossEntropyLoss(weight=cw_final)
optimizer_f = torch.optim.AdamW(final_model.parameters(), lr=lr, weight_decay=weight_decay)

# Retrain on train+val with a fixed budget (do not early-stop on the test set).
final_epochs = 100
for _ in range(final_epochs):
    train_one_epoch(final_model, trainval_loader, optimizer_f, criterion_f)

test_acc = validation_accuracy(final_model, test_loader)
print(f"Test accuracy: {test_acc:.4f}")

[I 2026-04-01 20:14:36,953] A new study created in memory with name: no-name-e37345df-cc4d-465f-a392-2ce0fdf0a146
  0%|          | 0/40 [02:33<?, ?it/s]

[W 2026-04-01 20:17:10,414] Trial 0 failed with parameters: {'n_layers': 4, 'n_units_0': 129, 'n_units_1': 590, 'n_units_2': 257, 'n_units_3': 193, 'lr': 5.9208440501203e-05, 'weight_decay': 0.0013984054295386162, 'dropout': 0.3173371263942386} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/Users/aryamanwade/Desktop/ml_prac/.venv/lib/python3.13/site-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "/var/folders/4w/gzh6xy110r9g8hzr2201cz3r0000gn/T/ipykernel_7844/3146730706.py", line 78, in objective
    train_one_epoch(model, train_loader, optimizer, criterion)
    ~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/var/folders/4w/gzh6xy110r9g8hzr2201cz3r0000gn/T/ipykernel_7844/3146730706.py", line 27, in train_one_epoch
    loss.backward()
    ~~~~~~~~~~~~~^^
  File "/Users/aryamanwade/Desktop/ml_prac/.venv/lib/python3.13/site-packages/torch/_tensor.py", line 631, in b

KeyboardInterrupt: 